[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/09_naive_bayes/90_exercises.ipynb)

# Bài tập — Chương 09: Naive Bayes (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/09_naive_bayes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import BernoulliNB, GaussianNB, MultinomialNB
from sklearn.pipeline import make_pipeline

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết định lý Bayes; quy tắc phân loại MAP của Naive Bayes trong không gian log; công thức Laplace smoothing cho Multinomial NB.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích cho một quản lý sản phẩm: vì sao bộ lọc spam Naive Bayes báo "99.99% là spam" cho rất nhiều email, kể cả những email thực ra chỉ "hơi giống" spam? Có nên dùng con số đó để quyết định xoá email tự động không?

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Một bài kiểm tra phát hiện gian lận: 0.5% giao dịch là gian lận; bài kiểm tra bắt được 90% gian lận và báo động nhầm 2% giao dịch bình thường. Tính $P(\text{gian lận} \mid \text{báo động})$.

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
p_fraud_given_alarm = None
### END CODE HERE ###

In [ ]:
check_close(p_fraud_given_alarm, 0.18443, tol=1e-4, message="P(gian lận | báo động)")

**Câu 4.** Multinomial NB, từ điển 4 từ `[giảm, giá, họp, lịch]`, $\alpha = 1$. Tổng đếm từ trong lớp spam: `[6, 4, 0, 0]`, lớp ham: `[1, 1, 5, 3]`. Prior: $P(\text{spam}) = 0.4$. Văn bản mới có đếm `[1, 1, 1, 0]`. Tính $\theta$ hai lớp (đã smoothing) và $P(\text{spam} \mid \mathbf x)$.

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
theta_spam = None
theta_ham = None
x_doc = None
s_spam, s_ham = None, None
p_spam = None
### END CODE HERE ###

In [ ]:
check_close(theta_spam, [0.5, 5 / 14, 1 / 14, 1 / 14], message="θ spam")
check_close(theta_ham, [1 / 7, 1 / 7, 3 / 7, 2 / 7], message="θ ham")
check_close(p_spam, 0.49296, tol=1e-4, message="P(spam | x) — sát nút!")

## L3 — Code

**Bài 5.** Viết `gaussian_nb_fit(X, y)` trả về `(classes, mu, var, log_prior)` và `gaussian_nb_predict_proba(X, mu, var, log_prior)` (tính trong không gian log, đổi về xác suất bằng `logsumexp`). Không cần `var_smoothing`.

In [ ]:
def gaussian_nb_fit(X, y):
    classes = np.unique(y)
    ### START CODE HERE ### (≈ 3 dòng)
    mu = None
    var = None
    log_prior = None
    ### END CODE HERE ###
    return classes, mu, var, log_prior


def gaussian_nb_predict_proba(X, mu, var, log_prior):
    ### START CODE HERE ### (≈ 3 dòng)
    log_pdf = None
    jll = None
    proba = None
    ### END CODE HERE ###
    return proba

In [ ]:
iris = pd.read_csv(repo_path("data", "sample", "iris.csv"))
Xi, yi = iris.iloc[:, :4].to_numpy(), iris.species.to_numpy()
classes, mu, var, log_prior = gaussian_nb_fit(Xi, yi)
ours = gaussian_nb_predict_proba(Xi, mu, var, log_prior)
ref = GaussianNB(var_smoothing=0).fit(Xi, yi).predict_proba(Xi)
check_close(ours, ref, tol=1e-6, message="khớp sklearn GaussianNB")

**Bài 6.** Viết `bernoulli_nb_log_likelihood(X, p)`: với `X` nhị phân $(n \times d)$ và `p` $(C \times d)$ là $P(x_j = 1 \mid y = c)$, trả về ma trận $(n \times C)$ của $\sum_j \left[x_j \log p_{cj} + (1 - x_j)\log(1 - p_{cj})\right]$ — **không vòng lặp**. Lưu ý: Bernoulli NB phạt cả từ **vắng mặt**.

In [ ]:
def bernoulli_nb_log_likelihood(X, p):
    ### START CODE HERE ### (≈ 1 dòng)
    ll = None
    ### END CODE HERE ###
    return ll

In [ ]:
Xbin = (rng.random((50, 6)) < 0.4).astype(float)
ybin = rng.integers(0, 2, 50)
bnb = BernoulliNB(alpha=1.0).fit(Xbin, ybin)
p_hat = np.exp(bnb.feature_log_prob_)
check_close(bernoulli_nb_log_likelihood(Xbin, p_hat) + bnb.class_log_prior_, bnb.predict_joint_log_proba(Xbin),
            tol=1e-8, message="khớp sklearn BernoulliNB")

## L4 — Debug

**Bài 7.** Bộ lọc spam dưới đây (1) trả về toàn `nan`/0 với văn bản dài, và (2) "chết" khi gặp từ chưa thấy trong một lớp. Sửa: tính trong không gian log và dùng Laplace smoothing ($\alpha = 1$).

In [ ]:
def spam_scores(X_counts, word_counts_per_class, priors):
    """X_counts: (n, V) đếm từ; word_counts_per_class: (C, V); priors: (C,). Trả về log-score (n, C)."""
    ### START CODE HERE ### (≈ 3 dòng)
    theta = word_counts_per_class / word_counts_per_class.sum(axis=1, keepdims=True)    # 🐛 không smoothing
    return np.prod(theta[None] ** X_counts[:, None, :], axis=2) * priors               # 🐛 tích → underflow
    ### END CODE HERE ###

In [ ]:
wc = np.array([[30.0, 0, 5, 10], [2, 20, 8, 0]])
long_doc = np.array([[400.0, 3, 50, 200]])           # văn bản dài, có từ thứ 2 (chưa thấy ở lớp 0)
sc = spam_scores(long_doc, wc, np.array([0.5, 0.5]))
expected = long_doc @ np.log((wc + 1) / (wc + 1).sum(1, keepdims=True)).T + np.log(0.5)
check_close(sc, expected, message="log-score đúng công thức (log + Laplace)")
check_true(sc[0, 0] - sc[0, 1] > 100, "lớp 0 thắng cách biệt lớn — không còn cảnh underflow 0 vs 0")

## L5 — Phân tích

**Bài 8.** Kiểm chứng "Naive Bayes phân loại tốt nhưng xác suất quá tự tin" trên `reviews_vi.csv` (target `label`): chia train/test, train `MultinomialNB` trên **unigram** và trên **unigram + bigram + trigram**. So sánh accuracy và log-loss; tính tỉ lệ dự đoán có xác suất > 0.99 hoặc < 0.01 (`frac_extreme`). Nhận xét vì sao thêm n-gram làm xác suất cực đoan hơn.

In [ ]:
from sklearn.metrics import accuracy_score, log_loss

reviews = pd.read_csv(repo_path("data", "sample", "reviews_vi.csv"))
tr, te = train_test_split(reviews, test_size=0.3, random_state=0, stratify=reviews.label)
results = {}
for ngram in [(1, 1), (1, 3)]:
    ### START CODE HERE ### (≈ 4 dòng)
    pipe = None
    p = None
    frac_extreme = None
    ### END CODE HERE ###
for k, (acc, ll, fe) in results.items():
    print(f"ngram {k}: accuracy = {acc:.4f} | log-loss = {ll:.4f} | % xác suất cực đoan = {fe:.0%}")

In [ ]:
check_greater(results[(1, 3)][2], results[(1, 1)][2], "n-gram làm xác suất cực đoan hơn")
check_less(abs(results[(1, 3)][0] - results[(1, 1)][0]), 0.02, "accuracy gần như không đổi")

✍️ Nhận xét:

## L6 — Thực tế

**Bài 9.** Xây bộ phân loại **3 lớp** (`sentiment`: pos/neg/neu) cho `reviews_vi.csv`: Pipeline `CountVectorizer` + Naive Bayes, chọn `ngram_range`, loại NB (Multinomial/Complement) và `alpha` bằng `GridSearchCV` (5-fold, `f1_macro`) trên tập train. Báo cáo macro-F1 trên tập test (`test_f1`) và in `classification_report`. Mục tiêu `test_f1 ≥ 0.98`.

In [ ]:
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import ComplementNB

tr3, te3 = train_test_split(reviews, test_size=0.3, random_state=0, stratify=reviews.sentiment)
### START CODE HERE ### (≈ 6 dòng)
pipe = None
grid = None
pred = None
test_f1 = None
### END CODE HERE ###
print(grid.best_params_)
print(classification_report(te3.sentiment, pred, digits=3))

In [ ]:
check_greater(test_f1, 0.98, "macro-F1 test ≥ 0.98")

## L7 — Nâng cao

**Bài 10.** **Naive Bayes trực tuyến (online learning).** Multinomial NB chỉ cần **bộ đếm** → cập nhật được từng lô dữ liệu mà không cần train lại từ đầu. Viết class `OnlineMultinomialNB` với `partial_fit(X, y, classes)` cộng dồn `class_count_`, `feature_count_` và `predict(X)`. Chứng minh: train qua 10 lô cho **đúng cùng** dự đoán với train một lần trên toàn bộ dữ liệu.

In [ ]:
class OnlineMultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.class_count_ = None

    def partial_fit(self, X, y, classes):
        if self.class_count_ is None:
            self.classes_ = np.asarray(classes)
            self.class_count_ = np.zeros(len(classes))
            self.feature_count_ = np.zeros((len(classes), X.shape[1]))
        ### START CODE HERE ### (≈ 3 dòng)
        pass
        ### END CODE HERE ###
        return self

    def predict(self, X):
        ### START CODE HERE ### (≈ 4 dòng)
        counts = None
        log_theta = None
        log_prior = None
        jll = None
        ### END CODE HERE ###
        return self.classes_[np.argmax(jll, axis=1)]

In [ ]:
vec = CountVectorizer(ngram_range=(1, 2)).fit(tr.text)
X_tr, X_te = vec.transform(tr.text), vec.transform(te.text)
y_tr = tr.label.to_numpy()
online = OnlineMultinomialNB()
for chunk in np.array_split(np.arange(X_tr.shape[0]), 10):
    online.partial_fit(X_tr[chunk], y_tr[chunk], classes=[0, 1])
batch_pred = MultinomialNB().fit(X_tr, y_tr).predict(X_te)
check_true(np.array_equal(online.predict(X_te), batch_pred), "10 lô online = train một lần")